# Crimes Analysis on US Populations
Project B.

Lets begin by adding and loading all the packages that will be useful along the notebook.

In [ ]:
using Pkg
# Ensure required packages (uncomment to install if needed)
# Pkg.add([
#     "MLJ", 
#     "MLJBase", 
#     "MLJModels", 
#     "MLJEnsembles", 
#     "MLJLinearModels", 
#     "DecisionTree", 
#     "MLJDecisionTreeInterface",
#     "NaiveBayes",
#     "MLJNaiveBayesInterface", 
#     "EvoTrees", 
#     "CategoricalArrays", 
#     "Random",
#     "LIBSVM",           
#     "MLJLIBSVMInterface", 
#     "Plots",
#     "MLJModelInterface",
#     "CSV",       
#     "DataFrames",      
#     "UrlDownload",
#     "XGBoost",
#     "MultivariateStats",
#     "MLJMultivariateStatsInterface",
#     "TSne",
#     "Optimisers",
#     "MLJFlux",
#     "NearestNeighborModels",
#     "Flux",
#     "Statistics",
#     "StatsPlots",
#     "PrettyTables",
#     "Printf"
# ])

In [ ]:
using MLJ;
using Plots;
using TSne;
using DataFrames;
using CSV;
using CategoricalArrays;
using Random;
using Pkg;
using LIBSVM;
using MLJFlux;
using Printf;
PCA = MLJ.@load PCA pkg=MultivariateStats
ICA = MLJ.@load ICA pkg=MultivariateStats
LDA = MLJ.@load LDA pkg=MultivariateStats verbosity=0
ICA = MLJ.@load ICA pkg=MultivariateStats
# TSNE = MLJ.@load TSNE pkg=TSne

LogisticClassifier = @load LogisticClassifier pkg=MLJLinearModels verbosity=0
GaussianNBClassifier = @load GaussianNBClassifier pkg=NaiveBayes verbosity=0


In [ ]:
# Finally lets import the functions created during the previous assignments
include("src/Preprocessing.jl")
include("src/Sampling.jl")
include("src/Training.jl")
include("src/Output.jl")
include("src/Ensembles.jl")


In [ ]:
using .Sampling
using .Preprocessing
using .Training
using .Output
using .Ensembles

## Loading data

The next phase is to load the data of our dataset and clean it. The dataset cames as two files the first one is a CSV with all the data (`communities.data`) and the second one (`communities.names`) gives information about the preprocessing being done.

The data cames with missing values which are marked as '?', these values where transformed into missing values to help the dataframe detect the type of each column.

Notice that we won't normalize the data as it has been previously normalize between 0 and 1 retaining their distribution and skew.

In [ ]:
Random.seed!(1234); # For reproducibility

# Laoding data
df = CSV.read("data/communities.data", DataFrame; header=false, missingstring="?");
df

After importing the data we change the name of the columns by its corresponding one and then we make a first quick analysis.

In [ ]:
#TODO: Add column names to CSV
header_string = "state, county, community, communityname, fold, population, householdsize, racepctblack, racePctWhite, racePctAsian, racePctHisp, agePct12t21, agePct12t29, agePct16t24, agePct65up, numbUrban, pctUrban, medIncome, pctWWage, pctWFarmSelf, pctWInvInc, pctWSocSec, pctWPubAsst, pctWRetire, medFamInc, perCapInc, whitePerCap, blackPerCap, indianPerCap, AsianPerCap, OtherPerCap, HispPerCap, NumUnderPov, PctPopUnderPov, PctLess9thGrade, PctNotHSGrad, PctBSorMore, PctUnemployed, PctEmploy, PctEmplManu, PctEmplProfServ, PctOccupManu, PctOccupMgmtProf, MalePctDivorce, MalePctNevMarr, FemalePctDiv, TotalPctDiv, PersPerFam, PctFam2Par, PctKids2Par, PctYoungKids2Par, PctTeen2Par, PctWorkMomYoungKids, PctWorkMom, NumIlleg, PctIlleg, NumImmig, PctImmigRecent, PctImmigRec5, PctImmigRec8, PctImmigRec10, PctRecentImmig, PctRecImmig5, PctRecImmig8, PctRecImmig10, PctSpeakEnglOnly, PctNotSpeakEnglWell, PctLargHouseFam, PctLargHouseOccup, PersPerOccupHous, PersPerOwnOccHous, PersPerRentOccHous, PctPersOwnOccup, PctPersDenseHous, PctHousLess3BR, MedNumBR, HousVacant, PctHousOccup, PctHousOwnOcc, PctVacantBoarded, PctVacMore6Mos, MedYrHousBuilt, PctHousNoPhone, PctWOFullPlumb, OwnOccLowQuart, OwnOccMedVal, OwnOccHiQuart, RentLowQ, RentMedian, RentHighQ, MedRent, MedRentPctHousInc, MedOwnCostPctInc, MedOwnCostPctIncNoMtg, NumInShelters, NumStreet, PctForeignBorn, PctBornSameState, PctSameHouse85, PctSameCity85, PctSameState85, LemasSwornFT, LemasSwFTPerPop, LemasSwFTFieldOps, LemasSwFTFieldPerPop, LemasTotalReq, LemasTotReqPerPop, PolicReqPerOffic, PolicPerPop, RacialMatchCommPol, PctPolicWhite, PctPolicBlack, PctPolicHisp, PctPolicAsian, PctPolicMinor, OfficAssgnDrugUnits, NumKindsDrugsSeiz, PolicAveOTWorked, LandArea, PopDens, PctUsePubTrans, PolicCars, PolicOperBudg, LemasPctPolicOnPatr, LemasGangUnitDeploy, LemasPctOfficDrugUn, PolicBudgPerPop, ViolentCrimesPerPop"
new_headers = [Symbol(strip(name)) for name in split(header_string, ',')]
rename!(df, Dict(names(df) .=> new_headers))
describe(df)

## Cleaning data

The next phase of our analysis is cleaning the data to delete unnecessary variables and those which add more noise that information. If possible we will try not to delete variables but instead try to infer missing values.

In the previous cell a first analysis was done, where we detected a number of columns with the same number of missing values as rows in the dataset. We decided we will begin this section identifying them, to analysis the reason of those missing values.

In [ ]:
# We detected a group of columns with the same number of missing values as rows in the dataset
n = 1675

# 2. Count the missing values for each column
missing_counts = map(col -> sum(ismissing, col), eachcol(df))
# Result: [1, 3, 0, 5, 1]

# 3. Find the indices where the count equals n
column_indices = findall(==(n), missing_counts)
# Result: [2] (Index for column B)

# 4. Get the column names using the indices
columns_with_n_missings = names(df)[column_indices]

We decided to remove these columns as they have to many missing values to infer them ($\approx 0.84\%$) and they are caused based on `communities.data` due to missing Law Enforcement Management and Administrative Statistics data on some communities.

In [ ]:
# Remove all columns with n missing values
select!(df, Not(columns_with_n_missings))

The next thing to do is identify the rest of the variables with missing values so we can manage this missing values.

In [ ]:
# See pending nulls columns
function describe_missing(df::DataFrame)
    
    pending_nulls = map(col -> sum(ismissing, col), eachcol(df))
    pending_nulls_dict = Dict(names(df) .=> pending_nulls)

    println("Columns with pending missing values:")
    for (col_name, miss_count) in pending_nulls_dict
        if miss_count > 0
            println("$col_name: $miss_count missing values")
        end
    end
end
describe_missing(df)

We identify the variable OtherPerCap which represents per capita income for people with heritage not coming from caucasians, african americans, native americans, asian, hispanic. As this is a numeric continuous variable, with only one missing values, we decided to infer the missing value instead of deleting the row.

To infer the missing value a mean was calculated and the value was replaced by this mean.

In [ ]:
# Complete OtherPerCap missing values with the mean of the column
mean_otherpercap = mean(skipmissing(df.OtherPerCap))
df.OtherPerCap .= coalesce.(df.OtherPerCap, mean_otherpercap)
describe_missing(df)

The next step was deleting other data identified as non predictive variables, which means that they are not useful to train our model to predict the goal variable.

In [ ]:
# Removing not predictive Variables
select!(df, Not([:state, :county, :community, :communityname, :fold]))

We continue by trying to find variables with all the values with the same value

In [ ]:
# Access LemasPctOfficDrugUn
# Show statistics for LemasPctOfficDrugUn I want mean, median, mode, std
# TODO Probably delete this column
lemas_col = df.LemasPctOfficDrugUn
mean_lemas = mean(skipmissing(lemas_col))
median_lemas = median(skipmissing(lemas_col))
mode_lemas = mode(skipmissing(lemas_col))
std_lemas = std(skipmissing(lemas_col))
println("Statistics for LemasPctOfficDrugUn:")
println("Mean: $mean_lemas")
println("Median: $median_lemas")
println("Mode: $mode_lemas")
println("Standard Deviation: $std_lemas")

In [ ]:
"""
    detect_values_near_0(df::DataFrame, threshold::Float64, count::Int=0)

Identifies columns in a DataFrame where the number of values whose absolute 
magnitude is less than or equal to a given 'threshold' meets a specified 'count'.

# Arguments
- `df::DataFrame`: The input DataFrame.
- `threshold::Float64`: The maximum absolute value for a number to be considered "near 0".
- `count::Int=0`: The required number of "near 0" values for a column to be selected.

# Returns
- `near_0_cols::Vector{Symbol}`: A list of column names (Symbols) that meet the criteria.
"""
function detect_values_near_0(df::DataFrame, threshold::Float64, count::Int=0, verbose::Bool=true)::Vector{Symbol}
    
    # 1. Initialize variables and prepare for reporting
    near_0_cols = Symbol[]
    report = "Value Detection Report (Threshold: $threshold, Required Count: $count)\n"
    
    # 2. Iterate through each column in the DataFrame
    for col_name in names(df)
        col = df[!, col_name]
        
        # Check if the column elements are of a numeric type (Int, Float, etc.)
        if eltype(col) <: Number
            
            # Count how many values are near zero
            near_0_value_count = sum(abs.(col) .<= threshold)
            
            # Add to the report
            report *= "* Column `:$col_name`: Found $near_0_value_count values near 0.\n"
            
            # Check the criteria
            if near_0_value_count >= count
                push!(near_0_cols, Symbol(col_name))
            end
        else
            # Skip non-numeric columns and note it in the report
            report *= "* Column `:$col_name`: Skipped (Non-numeric Type: $(eltype(col))).\n"
        end
    end
    if verbose
        # 3. Print the full report
        println(report)
    end
    
    # 4. Final summary for the return value
    if isempty(near_0_cols)
        println("Result: No numeric columns matched the criteria (count == $count).")
    else
        println("Result: The following columns matched the criteria: $near_0_cols")
    end

    # 5. Return the list of matching column symbols
    return near_0_cols
end


In [ ]:
len_df = size(df)[1]
n= 0.3* len_df
# Cast n to Int
n = round(Int, n)
near_0_columns = detect_values_near_0(df, 0.00001, n, true)

println("Columns detected with near 0 values $(length(near_0_columns)):")
for col in near_0_columns
    println("Column to consider for removal due to near 0 values: $col")
end

Tendremos esto en cuenta en futuros análisis, viendo el impacto de estas variables o su ausencia.

In [ ]:
df_without_near0 = select(df, Not(near_0_columns))

Our next step is to detect unvariant variables, those without variation in the whole dataset on their values. If found they will be deleted as they won't give any information for the regression but they will add noise.

In [ ]:

"""
    detect_unvariant_variables(df::DataFrame)

Identifies columns in a DataFrame where all rows have the same value (i.e., the column is unvariant).

# Arguments
- `df::DataFrame`: The input DataFrame.

# Returns
- `unvariant_cols::Vector{Symbol}`: A list of column names (Symbols) that are unvariant.
"""
function detect_unvariant_variables(df::DataFrame; verbose::Bool=true)::Vector{Symbol}
    
    # 1. Initialize variables and prepare for reporting
    unvariant_cols = Symbol[]
    report = "Unvariant Variable Detection Report\n"
    
    # 2. Iterate through each column in the DataFrame
    for col_name in names(df)
        col = df[!, col_name]
        
        # Get the number of unique values in the column
        n_unique = length(unique(col))
        
        # Add to the report
        report *= "* Column `:$col_name`: Found $n_unique unique value(s). (Type: $(eltype(col)))\n"
        
        # Check the criteria: A column is unvariant if it has only 1 unique value
        if n_unique == 1
            push!(unvariant_cols, Symbol(col_name))
        end
    end
    
    if verbose
        # 3. Print the full report
        println(report)
    end
    # 4. Final summary for the return value
    if isempty(unvariant_cols)
        println("Result: No columns were found to be unvariant.")
    else
        println("Result: The following columns were found to be unvariant: $unvariant_cols")
    end

    # 5. Return the list of matching column symbols
    return unvariant_cols
end
detect_unvariant_variables(df; verbose=false)

Finally we will make a fasta analysis in search of somethig wierd.

In [ ]:
function fast_analysis(df::DataFrame, columns::Vector{Symbol})
    println("Fast Analysis Report:")
    stats = zeros(Float64, length(columns), 7)
    for i in 1:length(columns)
        col_name = columns[i]
        col = df[!, col_name]
        mean_val = mean(skipmissing(col))
        median_val = median(skipmissing(col))
        std_val = std(skipmissing(col))
        min = minimum(skipmissing(col))
        max = maximum(skipmissing(col))
        unique_vals = length(unique(skipmissing(col)))
        missing_count = sum(ismissing, col)

        stats[i,:] = [mean_val, median_val, std_val, min, max, unique_vals, missing_count]
        
        println("Column: $col_name")
        println("  Min: $min")
        println("  Max: $max")
        println("  Mean: $mean_val")
        println("  Median: $median_val")
        println("  Standard Deviation: $std_val")
        println("  Unique Values: $unique_vals")
        println("  Missing Values: $missing_count")
        
    end

    return DataFrame(stats, :auto)
end
col_names = propertynames(df)
df_analysis = fast_analysis(df, col_names)


In [ ]:
column_names = ["mean_val", "median_val", "std_val", "min", "max", "unique_vals", "missing_count"]
rename!(df_analysis, column_names)
insertcols!(df_analysis, 1, :Feature_name => col_names)
df_analysis

In [ ]:
CSV.write("metrics.csv", df_analysis)

In [ ]:
df_limpio, matriz_corr, nombres = reduce_features_by_causality(df, :ViolentCrimesPerPop, umbral=0.8)

heatmap(
    nombres, String.(nombres), matriz_corr,
    title="Matriz de Correlación de Pearson",
    c=cgrad([:blue, :white, :red]),
    yflip=true,
    xrotation=90,
    size=(800, 800)
)

In [ ]:
df_limpio

## Data Preparation & Dimension Reduction

These finalize the cleaning of the data. From now on we will prepare the data to be passed to training models, then we will divide the data, and try to reduce its dimensionallity. 

First lets prepare the data to be ready for training ML models.

In [ ]:
using .Sampling
data = Matrix(df)
data_without_near0 = Matrix(df_without_near0)
data_corr = Matrix(df_limpio)
train_idx, test_idx = Sampling.holdOut(size(data, 1), 0.2)

train_input = data[train_idx, 1:end-1]
train_output = data[train_idx,end]
test_input = data[test_idx, 1:end-1]
test_output = data[test_idx, end]

train_corr = data_corr[train_idx, 1:end-1]
test_corr = data_corr[test_idx, 1:end-1]

In [ ]:
using Optimisers
using Flux

ann_hyperparameters = Dict(
    :topology => (64, 32),
    :transferFunctions => Flux.relu,
    :epochs => 1000,
    :optimiser => Optimisers.Adam(0.01),
    :batch_size => 32,
    :finaliser => identity, # identity is correct because your original buildClassANN handles the final activation (softmax or σ)
    :loss => Flux.mse   
)

model = createModel(:ANN, ann_hyperparameters)


In [ ]:
mach = machine(model, MLJ.table(train_input), categorical(train_output))

Now we will focus on two dimensionallity reduction techniques, PCA and ICA. We discarded LDA because it is a technique that depends on classes and here we are dealing with a regression problem.

First we go with PCA, we will be creating to models and so to datasets one with selecting a total variance of 95% and the other with a total variance of 99%.

In [ ]:
# PCA keeping 95% of the total variance
pca_model = PCA(variance_ratio=0.95)

# Train the PCA model
pca_mach = machine(pca_model, MLJ.table(train_input))
MLJ.fit!(pca_mach)

# Transform the data
pca_train = MLJ.matrix(MLJ.transform(pca_mach, MLJ.table(train_input)))
pca_test = MLJ.matrix(MLJ.transform(pca_mach, MLJ.table(test_input)))

print("Train Patterns ", size(train_input), " -> ", size(pca_train), "\n")
print("Test Patterns ", size(test_input), " -> ", size(pca_test), "\n")

In [ ]:
# PCA keeping 99% of the total variance
pca_model_99 = PCA(variance_ratio=0.99)

# Train the PCA model
pca_mach_99 = machine(pca_model_99, MLJ.table(train_input))
MLJ.fit!(pca_mach_99)

# Transform the data
pca_train_99 = MLJ.matrix(MLJ.transform(pca_mach_99, MLJ.table(train_input)))
pca_test_99 = MLJ.matrix(MLJ.transform(pca_mach_99, MLJ.table(test_input)))

print("Train Patterns ", size(train_input), " -> ", size(pca_train_99), "\n")
print("Test Patterns ", size(test_input), " -> ", size(pca_test_99), "\n")

As we can see mantaining 99% of the variance we have already deleted halve of the variables, menaing that they doesn't add any information, only 1% of variance spread in more than 40 variables.

After multiple tries we show next that ICA only finish reducing the model with output dimensions = 2 for this dataset, what is useles because is sure that a lot of information will be lost.

In [ ]:
# for seed in 1:100
#     Random.seed!(seed);

#     try
#         ica_model = ICA(outdim=10, tol=0.1)
#         ica_mach = machine(ica_model, MLJ.table(train_input))
#         fit!(ica_mach)

#         ica_train = MLJ.matrix(MLJ.transform(ica_mach, MLJ.table(train_input)))
#         ica_test = MLJ.matrix(MLJ.transform(ica_mach, MLJ.table(test_input)))

#         println("Train Patterns ", size(train_input), " -> ", size(ica_train))
#         println("Test Patterns ", size(test_input), " -> ", size(ica_test))
#         print("Seed: ", seed)
#         break
#     catch e
#         println("An error occurred with seed $seed: $e")
#     end
# end
# Is left commented as it prints lot of trash.

In [ ]:
ica_model = ICA(outdim=2, tol=0.3)
ica_mach = machine(ica_model, MLJ.table(train_input))
MLJ.fit!(ica_mach)

ica_train = MLJ.matrix(MLJ.transform(ica_mach, MLJ.table(train_input)))
ica_test = MLJ.matrix(MLJ.transform(ica_mach, MLJ.table(test_input)))

println("Train Patterns ", size(train_input), " -> ", size(ica_train), "\n")
println("Test Patterns ", size(test_input), " -> ", size(ica_test), "\n")


Now let's make a quick look into the dependant variable to see it's distribution. 

In [ ]:
outputs = data[:, end]
histogram(outputs, 
    title="Histogram of Outputs",
    xlabel="Value",
    ylabel="Frequency"
)


In [ ]:
using .Sampling
using .Preprocessing
using .Training
using .Output
using .Ensembles

In [ ]:
# edges = [0.15, 0.4, 0.70, 1.0, 6.0]
# binned_vector = bin_to_multiclass(outputs, edges)
# # Stratified binarization logaritmic
# println("Original Data: ", outputs)
# println("Bin Edges: ", edges)
# println("Binned Vector (Bin Indices): ", binned_vector)

using Statistics

# using quantile, we obtain an equal distribution of samples per bin
raw_target = data[:, end]
edges = quantile(raw_target, range(0, 1, length=5))
edges[end] += 0.0001
println("Bin Edges: ", edges)

binned_vector = Preprocessing.bin_to_multiclass(raw_target, edges)

counts = [count(x -> x == i, binned_vector) for i in 1:4]
println("Distribution of classes: ", counts)

In [ ]:
train_output_binned = binned_vector[train_idx,end]
test_output_binned = binned_vector[test_idx, end]

Como resultado del procesamiento de los datos se ofrecen:

- Como datos de entrada: 
    - Sin reducir: train_input, val_input, test_input
    - Con PCA (0.95 of total variance, 33 features): pca_train, pca_val, pca_test
    - Con PCA (0.99 of total variance, 58 features): pca_train_99, pca_val_99, pca_test_99
- Como datos de salida:
    - Regression: train_output, val_output, test_output
    - Classification (applying bining): train_output_binned, val_output_binned, test_output_binned

Here we do a training based on k-fold cross validation, checking different configurations of the classification models used in class.

The idea is to check what of those configurations present the best accuracy and standard deviation for further use afterwards

In [ ]:
using Printf

"""
    ExperimentSuite

Estructura mutable que encapsula todo el estado de un experimento de Machine Learning.
Mantiene los datos, la configuración y los resultados de los modelos entrenados.

# Models Config Update
Para usar con `Training.modelCrossValidation`, `models_config` debe ser un diccionario:
`"Nombre" => (:SimboloModelo, [ListaDeDiccionariosDeParametros])`
Ejemplo:
`"ANN" => (:ANN, [Dict(:topology => [8], ...), ...])`
"""
mutable struct ExperimentSuite
    X                               # Features
    y                               # Target (ya procesado)
    x_test
    y_test
    task_type::Symbol               # :regression o :classification
    split_strategy::ResamplingStrategy # Estrategia de validación (Holdout, CV, etc.)
    models_config::Dict             # Diccionario de Modelos y sus Rangos
    results::Dict                   # Almacenará los reportes y métricas
    best_machines::Dict             # Almacenará las máquinas entrenadas (objetos fit)
    
    """
    Constructor del ExperimentSuite.
    Recibe los datos de entrada (X) y salida (y) directamente como matrices o vectores.
    Realiza el preprocesamiento automático del target (binning) si es clasificación.
    """
    function ExperimentSuite(X_raw, y_raw, X_test_raw, y_test_raw, task_type::Symbol; 
                             n_classes::Int=0, split_strategy=Holdout(fraction_train=0.7), models_config=Dict())
        

        # 1. Preprocesamiento del Target según la tarea
        if task_type == :classification
            if n_classes > 0
                # Discretización basada en cuantiles para convertir regresión continua en clasificación
                println("Transformando target continuo a $n_classes clases usando cuantiles...")
                # Usamos una lógica simple de cuantiles manuales para asegurar compatibilidad
                q_vals = quantile(y_raw, range(0, 1, length=n_classes+1))
                # Ajuste pequeño para incluir extremos
                q_vals[1] -= 1e-5
                q_vals[end] += 1e-5
                
                # Función para binning
                function get_class(val, quantiles)
                    for i in 1:(length(quantiles)-1)
                        if val > quantiles[i] && val <= quantiles[i+1]
                            return i
                        end
                    end
                    return length(quantiles)-1 # Fallback
                end
                
                y_binned = [get_class(v, q_vals) for v in y_raw]
                y_test_binned = [get_class(v, q_vals) for v in y_test_raw]
                y_test = coerce(y_test_binned, OrderedFactor)
                y = coerce(y_binned, OrderedFactor)
            else
                y = coerce(y_raw, Multiclass)
                y_test = coerce(y_test_raw, Multiclass)
            end
        else
            # Regresión
            y = coerce(y_raw, Continuous)
            y_test = coerce(y_test_raw, Continuous)
        end

        # Coerce de Features (heurística simple, se puede mejorar con scitype explícito)
        # Importante: Usamos Continuous para evitar problemas con KNN/SVM y categorías inferidas
        X = coerce(X_raw, Continuous)
        X_test = coerce(X_test_raw, Continuous)
    

        new(X, y, X_test, y_test, task_type, split_strategy, models_config, Dict(), Dict())
    end
end

"""
    run_experiments!(suite::ExperimentSuite)

Ejecuta los experimentos utilizando `Training.modelCrossValidation` y `Sampling.crossvalidation`.
Itera sobre las configuraciones definidas en `suite.models_config`.
"""
function run_experiments!(suite::ExperimentSuite; measure=nothing)
    println("Iniciando ejecución de experimentos para tarea: $(suite.task_type)")
    println("==================================================================================================")
    println("Model           | Configuration                        | Result (CV Mean ± SD)")
    println("--------------------------------------------------------------------------------------------------")
    
    # 1. Preparar Dataset Tuple para Training module (Matriz Float32, Vector)
    dataset_tuple = (suite.X, suite.y)

    # 2. Generar Índices de CrossValidation usando Sampling
    k_folds = 10
    if suite.split_strategy isa CV
        k_folds = suite.split_strategy.nfolds
    end
    
    cv_indices = Sampling.crossvalidation(suite.y, k_folds)

    # 3. Iterar sobre Modelos y Configuraciones
    for (model_name, (model_symbol, param_configs)) in suite.models_config
        # model_symbol: Símbolo esperado por Training (ej. :ANN, :SVC)
        # param_configs: Vector de diccionarios con los parámetros a probar
        
        best_acc = -1.0
        best_result_obj = nothing
        
        for params in param_configs
            # Llamada al módulo Training original
            # modelCrossValidation retorna (results, mach) o similar
            res_obj = Training.modelCrossValidation(model_symbol, params, dataset_tuple, cv_indices)
            
            # Extraemos 'results' si es una tupla (results, mach)
            results = isa(res_obj, Tuple) ? res_obj[1] : res_obj
            
            # Obtener métricas para logging (Accuracy o Error)
            acc_mean = 0.0
            acc_std = 0.0
            
            # Detectar métrica disponible
            if hasproperty(results, :accuracy)
                acc_mean = results.accuracy[1]
                acc_std = results.accuracy[2]
            elseif hasproperty(results, :rms)
                    acc_mean = results.rms[1] # Placeholder si fuera regresión
            end
            
            # Formatear configuración para display
            config_str = string(params)
            if length(config_str) > 35
                config_str = config_str[1:32] * "..."
            end
            
            # Log en consola estilo tabla
            @printf("%-15s | %-35s | Acc: %.2f%% ± %.2f%%\n", 
                    model_name, config_str, acc_mean * 100, acc_std * 100)
            
            # Guardar el mejor resultado localmente
            if acc_mean > best_acc
                best_acc = acc_mean
                best_result_obj = (res_obj,model_symbol, params)
            end
                
            
        end
        
        # 4. Almacenar el mejor resultado en la Suite
        if best_result_obj !== nothing
            # Desempaquetar si es tupla (results, mach)
            final_res = best_result_obj[1]
            if best_result_obj[2] == :ANN
                params = deepcopy(best_result_obj[3])
                classes = sort(unique(suite.y))
                num_classes = length(classes)

                # OneHotEncoding
                one_hot_targets = oneHotEncoding(suite.y, classes)
                pop!(params, :validationRatio)
                ann, _, _, _ = trainClassANN(
                    pop!(params, :topology),
                    (suite.X, one_hot_targets);
                    params...
                )
                final_mach = ann
            else
                final_model = Training.createModel(best_result_obj[2], best_result_obj[3])
                final_mach = machine(final_model, MLJ.table(suite.X), categorical(suite.y))
                MLJ.fit!(final_mach, rows=1:size(suite.X, 1), verbosity=0)
            end
            

            
            suite.results[model_name] = (
                metric_score = best_acc,
                report = final_res,
                params = best_result_obj[3],
                model_type = best_result_obj[2],
                history = nothing # No disponible en legacy mode
            )
            
            if final_mach !== nothing

                suite.best_machines[model_name] = final_mach
            end
        end
    end
    println("--------------------------------------------------------------------------------------------------")
end

"""
    visualize_results(suite::ExperimentSuite; type=:bar, model_name=nothing)

Realiza predicciones sobre el conjunto de TEST (`suite.x_test`) usando los modelos entrenados
y genera visualizaciones según el tipo solicitado.

Tipos soportados (`type`):
- `:bar`: Gráfico de barras comparando el rendimiento (Accuracy o RMSE) de todos los modelos en Test.
- `:heatmap`: Matriz de confusión en Test (requiere `model_name` y `task_type=:classification`).
- `:scatter`: Gráfico de Predicción vs Real en Test (requiere `model_name` y `task_type=:regression`).
"""
function visualize_results(suite::ExperimentSuite; type=:bar, model_name=nothing)
    X_test_table = MLJ.table(suite.x_test)
    y_test = suite.y_test

    # -----------------------------------------------------------
    # 1. BAR CHART
    # -----------------------------------------------------------
    if type == :bar
        println("Generando comparativa de rendimiento en Test Set...")
        
        model_names = collect(keys(suite.best_machines))
        scores = Float64[]
        
        for name in model_names
            mach = suite.best_machines[name]
            model_type = suite.results[name].model_type
            
            if suite.task_type == :classification
                if model_type == :ANN
                    y_hat = mach(suite.x_test')
                    classes = sort(unique(y_test))
                    num_classes = length(classes)
                    one_hot_targets = oneHotEncoding(y_test, classes)
                    y_hat = classifyOutputs(y_hat')
                    # println(y_hat)
                    # println(one_hot_targets)
                    acc = Training.accuracy(y_hat, one_hot_targets)
                elseif model_type == :SVC
                    y_hat = MLJ.predict(mach, X_test_table)
                    acc = MLJ.accuracy(y_hat, y_test)
                else
                    y_hat = MLJ.predict_mode(mach, X_test_table)
                    acc = MLJ.accuracy(y_hat, y_test)
                end
                println(name, acc)
                push!(scores, acc)
            else
                y_hat = MLJ.predict(mach, X_test_table)
                err = MLJ.rmse(y_hat, y_test)
                println(name, err)
                push!(scores, err)
            end
        end
        
        perm = sortperm(scores)
        model_names = model_names[perm]
        scores = scores[perm]
        
        title_str = suite.task_type == :regression ? "Test RMSE (Menor es mejor)" : "Test Accuracy (Mayor es mejor)"
        y_lims = suite.task_type == :regression ? (0, maximum(scores))  : (0.3, maximum(scores))
        p = bar(model_names, scores, 
            orientation=:v, title=title_str, legend=false, color=:tab10,
            xrotation=45, ylabel="Score en Test", fmt=:png, ylims=y_lims)
        display(p)

    # -----------------------------------------------------------
    # 2. HEATMAP (Cálculo Manual)
    # -----------------------------------------------------------
    elseif type == :heatmap
        if suite.task_type != :classification
            println("El heatmap es solo para clasificación.")
            return
        end
        if isnothing(model_name)
            error("Debes especificar 'model_name'.")
        end
        
        mach = suite.best_machines[model_name]
        model_type = suite.results[model_name].model_type
        # println(model_name, model_type)
        # 1. Obtener predicciones y reales
        
        if model_type == :ANN
            outputs = mach(suite.x_test')
            outputs = classifyOutputs(outputs')
            outputs = [argmax(row) for row in eachrow(outputs)]
        elseif model_type == :SVC
            outputs = MLJ.predict(mach, X_test_table)
        else
            outputs = MLJ.predict_mode(mach, X_test_table) # Predicciones
        end
        targets = y_test                           # Reales
        
        # 2. Obtener clases (usamos levels de MLJ para asegurar el dominio completo)
        classes = MLJ.levels(targets)

        # 3. Lógica manual de ordenamiento (tu código)
        sorted_classes = try
            sort(classes, by = x -> parse(Float64, string(x)))
        catch
            sort(classes)
        end

        # 4. Construcción manual de la matriz (tu código)
        n = length(sorted_classes)
        class_map = Dict(c => i for (i, c) in enumerate(sorted_classes))
        matrix = zeros(Int, n, n)

        for (pred, real) in zip(outputs, targets)
            if haskey(class_map, pred) && haskey(class_map, real)
                r = class_map[real] # Fila: Real
                c = class_map[pred] # Columna: Predicho
                matrix[r, c] += 1
            end
        end

        # 5. Visualización
        labels_str = string.(sorted_classes)
        max_val = maximum(matrix)
        
        # Generar anotaciones con contraste de color
        anns = []
        for i in 1:n, j in 1:n
            val = matrix[i, j]
            # Texto blanco sobre fondo oscuro, negro sobre claro
            text_col = (val < max_val / 2) ? :white : :black
            push!(anns, (j, i, text(string(val), 10, text_col, :center, :center)))
        end

        p = heatmap(
            matrix,
            title="Matriz de Confusión: $model_name",
            xlabel="Predicho", 
            ylabel="Real",
            color = :blues,
            c=:viridis,
            aspect_ratio=:equal,
            xticks=(1:n, labels_str),
            yticks=(1:n, labels_str),
            yflip=true, # Fila 1 arriba
            annotations=anns
        )
        
        display(p)

    # -----------------------------------------------------------
    # 3. SCATTER
    # -----------------------------------------------------------
    elseif type == :scatter
        if suite.task_type != :regression
            println("El scatter plot es para regresión.")
            return
        end
        if isnothing(model_name)
            error("Debes especificar 'model_name'.")
        end
        
        mach = suite.best_machines[model_name]
        y_hat = MLJ.predict(mach, X_test_table)
        
        p = scatter(y_hat, y_test,
            title="Predicción vs Real: $model_name",
            xlabel="Predicho", ylabel="Real",
            legend=:topleft, label="Datos", markeralpha=0.6, color=:blue
        )
        plot!(p, y_test, y_test, color=:red, linestyle=:dash, label="Ideal")
        display(p)
        
    else
        println("Tipo desconocido.")
    end
end


In [ ]:
# Asegúrate de tener cargado el módulo MLTools

# using Flux # Necesario para la configuración de ANN (relu)

# ==========================================
# 1. DEFINICIÓN DE LA CONFIGURACIÓN (MODELS CONFIG)
# ==========================================
# Adaptamos los rangos al formato que espera Training.modelCrossValidation:
# "Nombre" => (:SimboloModelo, [Lista de Diccionarios de Parámetros])

# --- k-Nearest Neighbors ---
# K = 1, 3, 5, 7, 11, 21
knn_configs = [Dict(:K => k) for k in [1, 3, 5, 7, 11, 21]]

# --- Decision Tree ---
# Depths = 3, 5, 7, 9, 15, 20
dt_configs = [Dict(:max_depth => d) for d in [3, 5, 7, 9, 15, 20]]

# --- SVM ---
svm_configs = [
    Dict(:kernel => "LINEAR", :C => 0.1),
    Dict(:kernel => "LINEAR", :C => 1.0),
    Dict(:kernel => "LINEAR", :C => 10.0),

    Dict(:kernel => "RBF", :C => 1.0, :gamma => 0.01),
    Dict(:kernel => "RBF", :C => 1.0, :gamma => 0.1),
    Dict(:kernel => "RBF", :C => 10.0, :gamma => 0.01),
    Dict(:kernel => "RBF", :C => 100.0, :gamma => 0.001),

    Dict(:kernel => "POLY", :C => 1.0, :degree => Int32(2)) 
]



# --- Artificial Neural Networks (ANN) ---
# Topologías variadas

ann_topologies = [[8,], [16,], [32,], [64,], [8, 8], [16, 16], [32, 32], [64, 64]]
# ann_topologies = [[8,]]
ann_configs = [
    Dict(
        :topology => topo,
        :transferFunctions => fill(Flux.relu, length(topo)),
        :maxEpochs => 500,  
        :learningRate => 0.01,
        :showText => false,
        :validationRatio => 0.1,
        :maxEpochsVal => 10
    ) for topo in ann_topologies
]

# DICCIONARIO MAESTRO
# Mapea Nombre -> (Simbolo para Training.jl, Lista de Configs)
my_models = Dict(
    "kNN"        => (:KNeighborsClassifier, knn_configs),
    "D.Tree"     => (:DecisionTreeClassifier, dt_configs),
    "SVM"        => (:SVC, svm_configs),
    "ANN"        => (:ANN, ann_configs)
)

# ==========================================
# 2. DATOS DE ENTRADA
# ==========================================
# Asumo que 'train_input', 'train_corr', etc. están en tu workspace
experiment_inputs = Dict(
    "Raw Attributes"          => Float32.(train_input),
    "Filtered by Correlation" => Float32.(train_corr), 
    "PCA (95% Var)"           => Float32.(pca_train),
    "PCA (99% Var)"           => Float32.(pca_train_99),
    "ICA (2 Comp)"            => Float32.(ica_train)
)
experiment_tests = Dict(
    "Raw Attributes"          => Float32.(test_input),
    "Filtered by Correlation" => Float32.(test_corr), 
    "PCA (95% Var)"           => Float32.(pca_test),
    "PCA (99% Var)"           => Float32.(pca_test_99),
    "ICA (2 Comp)"            => Float32.(ica_test)
)
target = train_output # Tu vector de salida

suites_4 = Dict(
)
# ==========================================
# 3. EJECUCIÓN
# ==========================================
println(" INICIANDO SUITE DE EXPERIMENTOS MODULARIZADA (Backend: Training.jl)")

for (dataset_name, X_data) in experiment_inputs
    println("\n=============================================")
    println(" Procesando Dataset: $dataset_name")
    
    # A. Instanciar la Suite
    # Nota: Pasamos 'my_models' que ya tiene el formato correcto para Training.jl
    suite = ExperimentSuite(
        X_data, 
        target,
        experiment_tests[dataset_name],
        test_output,
        :classification;
        n_classes = 4, # Cuantiles
        models_config = my_models
    )
    
    # B. Ejecutar Experimentos
    # Esto llamará a Training.modelCrossValidation internamente
    run_experiments!(suite)
    
    suites_4[dataset_name] = suite
    # C. Visualizar Resultados
    println("Resumen Gráfico para $dataset_name:")
    for (model_name, model_parameters) in my_models
        visualize_results(suite, type=:heatmap, model_name=model_name)
    end
end

In [ ]:
for (dataset_name, X_data) in experiment_inputs
    println("\n=============================================")
    println(" Procesando Dataset: $dataset_name")
    
    
    # C. Visualizar Resultados
    println("Resumen Gráfico para $dataset_name:")
    visualize_results(suites_4[dataset_name], type=:bar)
end


In [ ]:
println(" INICIANDO SUITE DE EXPERIMENTOS MODULARIZADA (Backend: Training.jl)")
suites_3 = Dict(
)
for (dataset_name, X_data) in experiment_inputs
    println("\n=============================================")
    println(" Procesando Dataset: $dataset_name")
    
    # A. Instanciar la Suite
    # Nota: Pasamos 'my_models' que ya tiene el formato correcto para Training.jl
    suite = ExperimentSuite(
        X_data, 
        target,
        experiment_tests[dataset_name],
        test_output,
        :classification;
        n_classes = 3, # Cuantiles
        models_config = my_models
    )
    
    # B. Ejecutar Experimentos
    # Esto llamará a Training.modelCrossValidation internamente
    run_experiments!(suite)
    
    suites_3[dataset_name] = suite
    # C. Visualizar Resultados
    println("Resumen Gráfico para $dataset_name:")
    for (model_name, model_parameters) in my_models
        visualize_results(suite, type=:heatmap, model_name=model_name)
    end
end

In [ ]:
for (dataset_name, X_data) in experiment_inputs
    println("Resumen Gráfico para $dataset_name:")
    visualize_results(suites_3[dataset_name], type=:bar)
end


In [ ]:
println(" INICIANDO SUITE DE EXPERIMENTOS MODULARIZADA (Backend: Training.jl)")
suites_2 = Dict(
)
for (dataset_name, X_data) in experiment_inputs
    println("\n=============================================")
    println(" Procesando Dataset: $dataset_name")
    
    # A. Instanciar la Suite
    # Nota: Pasamos 'my_models' que ya tiene el formato correcto para Training.jl
    suite = ExperimentSuite(
        X_data, 
        target,
        experiment_tests[dataset_name],
        test_output,
        :classification;
        n_classes = 2, # Cuantiles
        models_config = my_models
    )
    
    # B. Ejecutar Experimentos
    # Esto llamará a Training.modelCrossValidation internamente
    run_experiments!(suite)
    
    suites_2[dataset_name] = suite
    # C. Visualizar Resultados
    println("Resumen Gráfico para $dataset_name:")
    for (model_name, model_parameters) in my_models
        visualize_results(suite, type=:heatmap, model_name=model_name)
    end
end

In [ ]:
for (dataset_name, X_data) in experiment_inputs
    
    println("Resumen Gráfico para $dataset_name:")
    visualize_results(suites_2[dataset_name], type=:bar)
end


In [ ]:
EpsilonSVR = @load EpsilonSVR pkg=LIBSVM verbosity=0
DTRegressor = @load DecisionTreeRegressor pkg=DecisionTree verbosity=0
KNNRegressor = @load KNNRegressor pkg=NearestNeighborModels verbosity=0
NeuralNetworkRegressor = @load NeuralNetworkRegressor pkg=MLJFlux verbosity=0

function run_experiments_regression!(suite::ExperimentSuite; measure=nothing)
    println("\n🚀 Iniciando ejecución de experimentos para tarea: $(suite.task_type)")
    println("==================================================================================================")
    println("Model           | Configuration                        | Result (CV RMSE ± SD)")
    println("--------------------------------------------------------------------------------------------------") 
    # 1. Preparar Dataset Tuple para Training module (Matriz Float32, Vector)
    current_targets_reg=Float32.(suite.y)
    inputs=Float32.(suite.X)
    # 2. Generar Índices de CrossValidation usando Sampling
    k_folds = 10
    cv_indices_reg = Sampling.crossvalidation(length(suite.y), k_folds)

    # 3. Iterar sobre Modelos y Configuraciones
    for (model_name, (model_symbol, param_configs)) in suite.models_config
        # model_symbol: Símbolo esperado por Training (ej. :ANN, :SVC)
        # param_configs: Vector de diccionarios con los parámetros a probar
        
        best_rmse = 100000.0
        best_std_rmse = 0
        best_result_obj = nothing
        mach = nothing
        for params in param_configs
                          

                rmse_folds = Float32[]
                for k in 1:k_folds
                    train_mask = cv_indices_reg .!= k
                    test_mask = cv_indices_reg .== k

                    if model_symbol == :ANN
                        model = NeuralNetworkRegressor(;params...)
                        pipe = Standardizer() |> model
                        mach = machine(pipe, MLJ.table(inputs[train_mask, :]), current_targets_reg[train_mask])

                    elseif model_symbol == :SVC
                        model = EpsilonSVR(;params...)
                        pipe = Standardizer() |> model
                        mach = machine(pipe, MLJ.table(inputs[train_mask, :]), current_targets_reg[train_mask])
                    
                    elseif model_symbol == :DecisionTreeClassifier
                        model = DTRegressor(;params...)
                        mach = machine(model, MLJ.table(inputs[train_mask, :]), current_targets_reg[train_mask])

                    elseif model_symbol == :KNeighborsClassifier
                        model = KNNRegressor(;params...)
                        mach = machine(model, MLJ.table(inputs[train_mask, :]), current_targets_reg[train_mask])

                    end
                    
                    MLJ.fit!(mach, verbosity=0)
                    preds = MLJ.predict(mach, MLJ.table(inputs[test_mask, :]))
                    
                    rmse, _ = evaluate_regression(preds, current_targets_reg[test_mask])
                    push!(rmse_folds, rmse)
                end
                
                # Formatear configuración para display
                config_str = string(params)
                if length(config_str) > 35
                    config_str = config_str[1:32] * "..."
                end
                rmse_mean = mean(rmse_folds)
                rmse_std = std(rmse_folds)
                # Log en consola estilo tabla
                @printf("%-15s | %-35s | RMSE: %.2f ± %.2f\n", 
                        model_name, config_str, rmse_mean, rmse_std)
                
                # Guardar el mejor resultado localmente
                
                if rmse_mean < best_rmse
                    best_rmse = rmse_mean
                    best_std_rmse = rmse_std
                    best_result_obj = ((best_rmse, best_std_rmse), model_symbol, params)
                end
                
        end
        
        # 4. Almacenar el mejor resultado en la Suite
        if best_result_obj !== nothing
            model_symbol = best_result_obj[2]
            final_res = best_result_obj[1]
            rmse = final_res[1]
            params = best_result_obj[3]
            mach = nothing
            if model_symbol == :ANN    
                # Instantiate model
                final_model = NeuralNetworkRegressor(;params...)
                final_pipe = Standardizer() |> final_model
                final_mach = machine(final_pipe, MLJ.table(inputs), suite.y)

            elseif model_symbol == :SVC
                final_model = EpsilonSVR(;params...)
                final_pipe = Standardizer() |> final_model
                final_mach = machine(final_pipe, MLJ.table(suite.X), suite.y)
            
            elseif model_symbol == :DecisionTreeClassifier
                final_model = DTRegressor(;params...)
                final_mach = machine(final_model, MLJ.table(suite.X), suite.y)

            elseif model_symbol == :KNeighborsClassifier
                final_model = KNNRegressor(;params...)
                final_mach = machine(final_model, MLJ.table(suite.X), suite.y)
            end
            MLJ.fit!(final_mach, rows=1:size(suite.X, 1), verbosity=0)
            
            suite.results[model_name] = (
                metric_score = best_rmse,
                report = final_res,
                model_type = model_symbol,
                history = nothing # No disponible en legacy mode
            )
            
            if final_mach !== nothing
                suite.best_machines[model_name] = final_mach
            end
        end
    end
    println("--------------------------------------------------------------------------------------------------")
end

In [ ]:

# --- k-Nearest Neighbors ---
# K = 1, 3, 5, 7, 11, 21
knn_configs = [Dict(:K => k) for k in [1, 3, 5, 7, 11, 21]]

# --- Decision Tree ---
# Depths = 3, 5, 7, 9, 15, 20
dt_configs = [Dict(:max_depth => d) for d in [3, 5, 7, 9, 15, 20]]

# --- SVM ---
svm_configs = [
    Dict(:kernel=>LIBSVM.Kernel.Linear, :cost=>1.0, :gamma=>0.1),
    Dict(:kernel=>LIBSVM.Kernel.Linear, :cost=>0.1, :gamma=>0.1),
    Dict(:kernel=>LIBSVM.Kernel.Linear, :cost=>10.0, :gamma=>0.1),
    Dict(:kernel=>LIBSVM.Kernel.RadialBasis, :cost=>1.0, :gamma=>0.01),
    Dict(:kernel=>LIBSVM.Kernel.RadialBasis, :cost=>1.0, :gamma=>0.1),
    Dict(:kernel=>LIBSVM.Kernel.RadialBasis, :cost=>10.0, :gamma=>0.01),
    Dict(:kernel=>LIBSVM.Kernel.RadialBasis, :cost=>100.0, :gamma=>0.001),
    Dict(:kernel=>LIBSVM.Kernel.Polynomial, :cost=>1.0, :gamma=>0.1, :degree=>Int32(2))

]


# --- Artificial Neural Networks (ANN) ---
# Topologías variadas

#ann_topologies = [[8,], [16,], [32,], [64,], [8, 8], [16, 16], [32, 32], [64, 64]]

ann_topologies = [(8,), (16,), (32,), (64,), (8,8), (16, 16), (32, 32), (64, 64)]

ann_epoch = [77, 20, 15, 7, 8, 8, 5, 5]

i = 1
ann_configs = []
for topo in ann_topologies
    current = Dict(
        :builder => MLJFlux.MLP(topo, Flux.relu),
        :epochs => ann_epoch[i],  
        :batch_size => 32,
        :optimiser => Optimisers.Adam(0.01)
    )
    
    push!(ann_configs, current)
    i = i + 1   
end

# DICCIONARIO MAESTRO
# Mapea Nombre -> (Simbolo para Training.jl, Lista de Configs)
my_models = Dict(
    "kNN"        => (:KNeighborsClassifier, knn_configs),
    "D.Tree"     => (:DecisionTreeClassifier, dt_configs),
    "SVM"        => (:SVC, svm_configs),
    "ANN"        => (:ANN, ann_configs)
)

# ==========================================
# 2. DATOS DE ENTRADA
# ==========================================
experiment_inputs = Dict(
    "Raw Attributes"          => Float32.(train_input),
    "Filtered by Correlation" => Float32.(train_corr), 
    "PCA (95% Var)"           => Float32.(pca_train),
    "PCA (99% Var)"           => Float32.(pca_train_99),
    "ICA (2 Comp)"            => Float32.(ica_train)
)

experiment_tests = Dict(
    "Raw Attributes"          => Float32.(test_input),
    "Filtered by Correlation" => Float32.(test_corr), 
    "PCA (95% Var)"           => Float32.(pca_test),
    "PCA (99% Var)"           => Float32.(pca_test_99),
    "ICA (2 Comp)"            => Float32.(ica_test)
)
target = train_output # Tu vector de salida

suites = Dict(
)
# ==========================================
# 3. EJECUCIÓN
# ==========================================
println(" INICIANDO SUITE DE EXPERIMENTOS MODULARIZADA (Backend: Training.jl)")

for (dataset_name, X_data) in experiment_inputs
    println("\n=============================================")
    println(" Procesando Dataset: $dataset_name")
    
    suite = ExperimentSuite(
        X_data, 
        target,
        experiment_tests[dataset_name],
        test_output,
        :regression;
        n_classes = 4, # Cuantiles
        models_config = my_models
    )
    
    # B. Ejecutar Experimentos
    run_experiments_regression!(suite)
    
    suites[dataset_name] = suite
    # C. Visualizar Resultados
    println("Resumen Gráfico para $dataset_name:")
    for (model_name, model_parameters) in my_models
        visualize_results(suite, type=:scatter, model_name=model_name)
    end
end

In [ ]:
for (dataset_name, X_data) in experiment_inputs
    
    println("Resumen Gráfico para $dataset_name:")
    visualize_results(suites[dataset_name], type=:bar)
end


In [ ]:
using DecisionTree

s = suites["Raw Attributes"]
# 1. Retrieve the names of the variables
# Exclude the last column because it is the target (ViolentCrimesPerPop)
feature_names = names(df)[1:end-1]

# Safety check:
println("Number of variables (Names): ", length(feature_names))
println("Number of columns in Train Input: ", size(s.X, 2))

mach_tree = s.best_machines["D.Tree"]

# 3. Extract Importance
# MLJ returns a vector like [:x1 => 0.3, :x5 => 0.1, ...]
raw_importances = feature_importances(mach_tree)

# 4. Map the generic indices (x1, x2...) to your Real Names
named_importances = []

for (feature_sym, score) in raw_importances
    # The symbol is like :x12. Convert it to string "x12"
    f_str = string(feature_sym)
    
    # Remove the "x" and convert the number to integer (e.g., "12" -> 12)
    idx = parse(Int, f_str[2:end])
    
    # Find the real name in the list we extracted from df
    real_name = feature_names[idx]
    
    push!(named_importances, (real_name, score))
end

# 5. Sort and Prepare Plot
sort!(named_importances, by = x -> x[2], rev=true) # From largest to smallest

# Take the Top 15
top_n = 15
top_data = named_importances[1:top_n]
labels = [x[1] for x in top_data]
values = [x[2] for x in top_data]


# 6. Print the list to the screen
println("\n=== TOP 10 PREDICTIVE VARIABLES ===")
for i in 1:10
@printf("%d. %-25s : %.4f%%\n", i, labels[i], values[i]*100)
end


In [ ]:
s = suites["Filtered by Correlation"]
# 1. Retrieve the names of the variables
# Exclude the last column because it is the target (ViolentCrimesPerPop)
feature_names = names(df_limpio)[1:end-1]

# Safety check:
println("Number of variables (Names): ", length(feature_names))
println("Number of columns in Train Input: ", size(s.X, 2))

mach_tree = s.best_machines["D.Tree"]

# 3. Extract Importance
# MLJ returns a vector like [:x1 => 0.3, :x5 => 0.1, ...]
raw_importances = feature_importances(mach_tree)

# 4. Map the generic indices (x1, x2...) to your Real Names
named_importances = []

for (feature_sym, score) in raw_importances
    # The symbol is like :x12. Convert it to string "x12"
    f_str = string(feature_sym)
    
    # Remove the "x" and convert the number to integer (e.g., "12" -> 12)
    idx = parse(Int, f_str[2:end])
    
    # Find the real name in the list we extracted from df
    real_name = feature_names[idx]
    
    push!(named_importances, (real_name, score))
end

# 5. Sort and Prepare Plot
sort!(named_importances, by = x -> x[2], rev=true) # From largest to smallest

# Take the Top 15
top_n = 15
top_data = named_importances[1:top_n]
labels = [x[1] for x in top_data]
values = [x[2] for x in top_data]


# 6. Print the list to the screen
println("\n=== TOP 10 PREDICTIVE VARIABLES ===")
for i in 1:10
@printf("%d. %-25s : %.4f%%\n", i, labels[i], values[i]*100)
end

After the cross validation, we have obtained the results of the different models with each configuration for each preprocessing strategy.
Here we are using the best for each preprocessing strategy with the whole training set.

And now, we are plotting the confussion matrix for the best models, to see which results are more interesting

This is an accuracy comparison for the aforementioned models

Now, we do the same as in classification but for regression. To do so, instead of the accuracy, we are checking the RMSE (or root mean squared error).

In [ ]:
include("src/Ensembles.jl")

In [ ]:
using .Ensembles

In [ ]:
using Statistics
using DataFrames, PrettyTables
using Plots
using StatsPlots
using Plots.Measures
gr()

function plotEnsembleConfusionsIndividual(
    ensemble_results::Dict{String, Tuple{Vector{String}, Vector{String}}},
    base_title::AbstractString;
    class_labels::Union{Nothing,Vector{String}} = nothing,
    save_dir::Union{Nothing,String} = nothing,
    file_prefix::AbstractString = ""
)
    # 1) Orden “bonito” de las representaciones
    preferred_order = [
        "Raw Attributes",
        "Filtered by Correlation",
        "PCA (95% Var)",
        "PCA (99% Var)",
        "ICA (2 Comp)"
    ]

    names = [n for n in preferred_order if haskey(ensemble_results, n)]
    if length(names) < length(ensemble_results)
        for n in sort(collect(keys(ensemble_results)))
            if !(n in names)
                push!(names, n)
            end
        end
    end

    # Si hay carpeta para guardar, asegúrate de que exista
    if save_dir !== nothing
        isdir(save_dir) || mkpath(save_dir)
    end

    for repr_name in names
        outputs, targets = ensemble_results[repr_name]

        # --- clases presentes en esa representación ---
        classes = unique(vcat(outputs, targets))

        # Orden numérico si se puede, si no alfabético
        sorted_classes = try
            sort(classes, by = x -> parse(Float64, string(x)))
        catch
            sort(classes)
        end

        n         = length(sorted_classes)
        coords    = 1:n
        class_idx = Dict(c => i for (i, c) in enumerate(sorted_classes))
        mat       = zeros(Int, n, n)

        # Matriz de confusión (filas = real, columnas = predicho)
        for (pred, real) in zip(outputs, targets)
            if haskey(class_idx, real) && haskey(class_idx, pred)
                r = class_idx[real]
                c = class_idx[pred]
                mat[r, c] += 1
            end
        end

        ann = [
            (j, i, text(string(mat[i, j]), 8, :black, :center, :center))
            for i in 1:n, j in 1:n
        ]

        # Etiquetas de clase “bonitas” si se pasan y cuadran en longitud
        display_labels =
            (class_labels !== nothing && length(class_labels) == n) ?
                class_labels :
                string.(sorted_classes)

        # Título de esta figura
        title_str = "$base_title – $repr_name"

        p = heatmap(
            coords,
            coords,
            mat;
            color         = :blues,
            aspect_ratio  = 1,
            xrotation     = 45,
            yflip         = true,
            xlabel        = "Predicted",
            ylabel        = "Real",
            title         = title_str,
            annotations   = vec(ann),
            xticks        = (coords, display_labels),
            yticks        = (coords, string.(sorted_classes)),
            size          = (700, 600)
        )

        display(p)

        if save_dir !== nothing
            safe_repr   = replace(repr_name, " " => "_", r"[^\w\-]" => "")
            safe_prefix = replace(file_prefix, " " => "_", r"[^\w\-]" => "")
            fname = isempty(safe_prefix) ?
                "cm_$(safe_repr).png" :
                "$(safe_prefix)_$(safe_repr).png"

            file_path = joinpath(save_dir, fname)
            savefig(p, file_path)
            println("Saved confusion matrix for '$repr_name' to: $file_path")
        end
    end

    return nothing
end

function plot_regression_ensemble_comparison(
    summary_hard,
    summary_equal,
    summary_weighted,
    summary_stacking,
    ordered_datasets::Vector{String};
    ensemble_labels = ["Hard Voting" "Equal Weights" "Weighted" "Stacking"],
    title::AbstractString = "Comparison of Regression Ensembles – Test RMSE",
    ylabel::AbstractString = "RMSE (Lower is better)",
    xlabel::AbstractString = "Data Strategy",
    save_path::Union{Nothing,String} = nothing
)

    # 2) Matriz (n_datasets x 4)
    final_data_ensembles = hcat(
        summary_hard.metrics,
        summary_equal.metrics,
        summary_weighted.metrics,
        summary_stacking.metrics
    )

    dataset_names = ordered_datasets

    p = groupedbar(
        final_data_ensembles;
        labels       = ensemble_labels,
        xticks       = (1:length(dataset_names), dataset_names),
        title        = title,
        ylabel       = ylabel,
        xlabel       = xlabel,
        bar_position = :dodge,
        bar_width    = 0.7,
        alpha        = 0.8,
        lw           = 0,
        size         = (900, 500),
        legend       = :topleft,
        color        = [:blue :orange :green :red]  
    )

    if save_path !== nothing
        # Crear directorios intermedios si no existen
        dir = dirname(save_path)
        if !isdir(dir)
            mkpath(dir)
        end

        savefig(p, save_path)
        println("Saved regression ensemble comparison plot to: $save_path")
    end

    display(p)
    return nothing
end


function print_reg_ensemble_summary(
    results::Dict{String, Tuple{Vector{Float64}, Vector{Float64}}},
    ordered_datasets::Vector{String};
    metric_fn,
    title::AbstractString,
    col_label::AbstractString
)
    # Use the helper from the Ensembles module to compute RMSE per representation
    summary = Ensembles.summarize_regression_ensembles(
        results,
        ordered_datasets;
        metric_fn = metric_fn
    )

    # Column name for the RMSE column (symbol for DataFrame)
    rmse_col_sym = Symbol(replace(col_label, " " => "_"))

    df = DataFrame(
        Representation = summary.representations,
        rmse_col_sym   = summary.metrics
    )

    println("\n$title")
    pretty_table(
        df,
        header     = ["Representation", col_label],
        formatters = ft_printf("%.4f", [2])
    )
    return df
end

function plotEnsembleAccuracyBestVsSimple(
    best_data,
    simple_data;
    folder_name::String   = "comparison_plots_best_vs_simple",
    base_filename::String = "ensemble_best_vs_simple",
    size                   = (900, 600),
    ylims                  = (0.0, 1.0)
)

    # Crear carpeta si no existe
    if !isdir(folder_name)
        mkdir(folder_name)
    end

    # Asumimos que best_data y simple_data vienen en el mismo orden de estrategias
    for i in eachindex(best_data)
        strategy_name_best, best_models   = best_data[i]
        strategy_name_simple, simple_models = simple_data[i]

        # sanity check rápido: mismo nombre de estrategia
        if strategy_name_best != strategy_name_simple
            @warn "Strategy name mismatch between best and simple data: " *
                  "best=$(strategy_name_best), simple=$(strategy_name_simple)"
        end

        strategy_name = strategy_name_best

        # Sacamos nombres de ensembles (Hard, Soft eq, Soft w, Stacking)
        ensemble_names_best   = [m[1] for m in best_models]
        ensemble_names_simple = [m[1] for m in simple_models]

        # Asumimos que el orden de ensembles coincide
        @assert ensemble_names_best == ensemble_names_simple "Ensemble order mismatch"

        ensemble_names = ensemble_names_best

        # Calculamos accuracies
        best_accs   = Float64[]
        simple_accs = Float64[]

        for j in eachindex(ensemble_names)
            # best
            _, preds_best, targets_best = best_models[j]
            acc_best = mean(string.(preds_best) .== string.(targets_best))
            push!(best_accs, acc_best)

            # simple
            _, preds_simple, targets_simple = simple_models[j]
            acc_simple = mean(string.(preds_simple) .== string.(targets_simple))
            push!(simple_accs, acc_simple)
        end

        # Matriz N_ensembles x 2 (Best, Simple)
        acc_matrix = hcat(best_accs, simple_accs)

        # Colores: más oscuro para Best, más claro para Simple
        base_colors = palette(:blues, 3)   # por ejemplo 3 tonos de azul
        color_best   = base_colors[end]    # más oscuro
        color_simple = base_colors[2]      # intermedio / más claro

        p = groupedbar(
            ensemble_names,
            acc_matrix;
            bar_position  = :dodge,
            bar_width     = 0.6,
            lw            = 0,
            xlabel        = "Ensemble",
            ylabel        = "Accuracy",
            title         = "Ensemble Accuracy – $(strategy_name)",
            ylims         = ylims,
            legend        = :topright,
            label         = ["Best models" "Simple models"],
            color         = [color_best color_simple],
            left_margin   = 25mm,
            bottom_margin = 20mm,
            size          = size
        )

        # Nombre de archivo seguro
        safe_name = replace(string(strategy_name), r"[^A-Za-z0-9_]+" => "_")
        filename  = "$(base_filename)_$(safe_name).png"
        save_path = joinpath(folder_name, filename)

        savefig(p, save_path)
        display(p)
    end
end

function plot_regression_ensemble_best_vs_simple(
    summary_hard_best,
    summary_equal_best,
    summary_weighted_best,
    summary_stacking_best,
    summary_hard_simple,
    summary_equal_simple,
    summary_weighted_simple,
    summary_stacking_simple,
    ordered_datasets;
    ensemble_labels::Vector{String} = ["Hard Voting", "Equal Weights", "Weighted", "Stacking"],
    folder_name::String             = "ensembles/regression/best_vs_simple",
    base_filename::String           = "regression_best_vs_simple",
    size                            = (900, 600),
    ylims                           = nothing  # ej. (0.4, 0.9) si quieres fijar escala
)

    if !isdir(folder_name)
        mkdir(folder_name)
    end

    n_strategies = length(ordered_datasets)

    for i in 1:n_strategies
        strategy_name = ordered_datasets[i]

        # RMSE de cada ensemble (Best)
        best_values = [
            summary_hard_best.metrics[i],
            summary_equal_best.metrics[i],
            summary_weighted_best.metrics[i],
            summary_stacking_best.metrics[i]
        ]

        # RMSE de cada ensemble (Simple)
        simple_values = [
            summary_hard_simple.metrics[i],
            summary_equal_simple.metrics[i],
            summary_weighted_simple.metrics[i],
            summary_stacking_simple.metrics[i]
        ]

        # Matriz 4 x 2 (4 ensembles, 2 barras: best vs simple)
        rmse_matrix = hcat(best_values, simple_values)

        # Colores: rojo oscuro para best, rojo más claro para simple
        base_colors  = palette(:reds, 3)
        color_best   = base_colors[end]
        color_simple = base_colors[2]

        if ylims === nothing
            p = groupedbar(
                ensemble_labels,
                rmse_matrix;
                bar_position  = :dodge,
                bar_width     = 0.6,
                lw            = 0,
                xlabel        = "Ensemble",
                ylabel        = "RMSE (Lower is better)",
                title         = "Regression Ensembles – $(strategy_name)",
                legend        = :topright,
                label         = ["Best models" "Simple models"],
                color         = [color_best color_simple],
                left_margin   = 25mm,
                bottom_margin = 20mm,
                size          = size
            )
        else
            p = groupedbar(
                ensemble_labels,
                rmse_matrix;
                bar_position  = :dodge,
                bar_width     = 0.6,
                lw            = 0,
                xlabel        = "Ensemble",
                ylabel        = "RMSE (Lower is better)",
                title         = "Regression Ensembles – $(strategy_name)",
                legend        = :topright,
                label         = ["Best models" "Simple models"],
                color         = [color_best color_simple],
                left_margin   = 25mm,
                bottom_margin = 20mm,
                size          = size,
                ylims         = ylims
            )
        end

        safe_name = replace(string(strategy_name), r"[^A-Za-z0-9_]+" => "_")
        filename  = "$(base_filename)_$(safe_name).png"
        save_path = joinpath(folder_name, filename)

        savefig(p, save_path)
        display(p)
    end
end

In [ ]:
using MLJFlux;
using Flux

# Simple ANN baseline (shallower + fewer epochs)
simple_ann_params = Dict(
    :topology => (8,),
    :transferFunctions => Flux.relu,
    :epochs => 100,
    :optimiser => Optimisers.Adam(0.01),
    :loss => Flux.mse,
    :batch_size => 32
)

simple_svm_params = Dict(
    :kernel => "LINEAR",   
    :C      => 1.0
)

simple_dt_params = Dict(
    :max_depth => 3        
)

simple_knn_params = Dict(
    :K => 3                
)

# Build the 
simple_ann_model = createModel(:ANN,                 simple_ann_params)
simple_svm_model = createModel(:SVPC,                 simple_svm_params)
simple_dt_model  = createModel(:DecisionTreeClassifier, simple_dt_params)
simple_knn_model = createModel(:KNeighborsClassifier,   simple_knn_params)

# One common list of base models for all feature representations
simple_base_models = [
    simple_ann_model,
    simple_svm_model,
    simple_dt_model,
    simple_knn_model
]

simple_ensemble_specs = Dict(
    "Raw Attributes" => (
        X_train = Float32.(train_input),
        X_test  = Float32.(test_input),
        models  = simple_base_models
    ),
    "PCA (95% Var)" => (
        X_train = Float32.(pca_train),
        X_test  = Float32.(pca_test),
        models  = simple_base_models
    ),
    "PCA (99% Var)" => (
        X_train = Float32.(pca_train_99),
        X_test  = Float32.(pca_test_99),
        models  = simple_base_models
    ),
    "ICA (2 Comp)" => (
        X_train = Float32.(ica_train),
        X_test  = Float32.(ica_test),
        models  = simple_base_models
    ),
    "Filtered by Correlation" => (
        X_train = Float32.(train_corr),
        X_test  = Float32.(test_corr),
        models  = simple_base_models
    )
)

In [ ]:
y_train_reg = suites["Raw Attributes"].y
y_test_reg  = suites["Raw Attributes"].y_test

# target = train_output # Tu vector de salida
function create_ensemble_specs(suites)
    reg_ensemble_specs = Dict()
    
    for (name, experiment) in pairs(suites)
        models = []
        
        # Itera sobre las máquinas/modelos dentro de cada experimento.
        for (names, mach) in experiment.best_machines
            
            if names == "SVM"
                # Si es "SVM", crea un modelo SVPC con los parámetros específicos de SVM.
                # Nota: Asumo que `experiment.results["SVM"]` es la estructura que contiene `params`.
                push!(models, Training.createModel(:SVPC, experiment.results["SVM"].params))
            elseif names == "ANN"
                continue
            else
                # En cualquier otro caso, usa el modelo de la máquina actual.
                push!(models, mach.model)
            end
        end
        
        # Construye la entrada del diccionario para el experimento actual.
        reg_ensemble_specs[name] = (
            X_train = experiment.X,
            X_test  = experiment.x_test,
            models  = models
        )
    end
    
    return reg_ensemble_specs
end
reg_ensemble_specs = Dict(
    name => (
        X_train = experiment.X,
        X_test  = experiment.x_test,
        models  = [
            mach.model
            for (names, mach) in experiment.best_machines 
        ]
    )
    for (name, experiment) in suites
)
reg_ensemble_specs = Dict{String, NamedTuple}(k => v for (k,v) in reg_ensemble_specs)


In [ ]:
using MLJFlux
using LIBSVM

# =========================================
# SIMPLE REGRESSION MODELS (for comparison)
# =========================================

# Simple ANN regressor (shallow, small topology)
simple_ann_reg_model = NeuralNetworkRegressor(
    builder    = MLJFlux.MLP((8,), Flux.relu),
    epochs     = 50,
    batch_size = 32,
    optimiser  = Optimisers.Adam(0.01)
)

# Simple SVR (linear kernel, modest C/gamma)
simple_svm_reg_model = EpsilonSVR(
    kernel = LIBSVM.Kernel.Linear,
    cost   = 1.0,
    gamma  = 0.1
)

# Simple Decision Tree Regressor (shallow tree)
simple_dt_reg_model = DTRegressor(
    max_depth = 3
)

# Simple kNN Regressor (small K)
simple_knn_reg_model = KNNRegressor(
    K = 3
)

# Common list of simple base models for all feature representations
simple_reg_base_models = [
    simple_ann_reg_model,
    simple_svm_reg_model,
    simple_dt_reg_model,
    simple_knn_reg_model
]

# =========================================
# SIMPLE REGRESSION ENSEMBLE SPECS
# =========================================

simple_reg_ensemble_specs = Dict(
    "Raw Attributes" => (
        X_train = train_input,
        X_test  = test_input,
        models  = simple_reg_base_models
    ),
    "PCA (95% Var)" => (
        X_train = pca_train,
        X_test  = pca_test,
        models  = simple_reg_base_models
    ),
    "PCA (99% Var)" => (
        X_train = pca_train_99,
        X_test  = pca_test_99,
        models  = simple_reg_base_models
    ),
    "ICA (2 Comp)" => (
        X_train = ica_train,
        X_test  = ica_test,
        models  = simple_reg_base_models
    ),
    "Filtered by Correlation" => (
        X_train = train_corr,
        X_test  = test_corr,
        models  = simple_reg_base_models
    )
)


In [ ]:
class_labels_4 = ["1-low crime \n(0.0, 0.07)", "2-low-medium crime \n(0.07, 0.15)", "3-medium-high crime \n(0.15, 0.33)", "4-high crime \n(0.33, 1.0)"]

exp_name = "4-class"

ordered_datasets = [
    "Raw Attributes",
    "PCA (95% Var)",
    "PCA (99% Var)",
    "ICA (2 Comp)",
    "Filtered by Correlation"
]

soft_weights_4 = [1.5, 2, 1.7]
soft_weights_4_simple = [1.5, 2, 1.3, 1.7]

ensemble_specs_4 =  create_ensemble_specs(suites_4)
ensemble_specs_4 = Dict{String, NamedTuple}(k => v for (k,v) in ensemble_specs_4)


In [ ]:
# ---------------------------
# 1) Hard voting (4-class)
# ---------------------------
hard_voting_4_results = Ensembles.run_voting_ensembles(
    ensemble_specs_4,
    suites_4["Raw Attributes"].y,
    suites_4["Raw Attributes"].y_test;
    voting       = :hard,
    title_prefix = "4-class crime"
)

hard_4_save_dir = joinpath("ensembles", exp_name, "hard_voting")
hard_4_file_prefix = "hard_voting_4class"

plotEnsembleConfusionsIndividual(
    hard_voting_4_results,
    "Hard Voting – ($exp_name)";
    class_labels = class_labels_4,
    save_dir     = hard_4_save_dir,
    file_prefix  = hard_4_file_prefix
)

In [ ]:
soft_equal_4_results = Ensembles.run_voting_ensembles(
    ensemble_specs_4,
    suites_4["Raw Attributes"].y,
    suites_4["Raw Attributes"].y_test;
    voting       = :soft,
    weights      = nothing,
    title_prefix = "4-class crime"
)

soft_equal_4_dir    = joinpath("ensembles", exp_name, "soft_equal_voting")
soft_equal_4_prefix = "soft_equal_voting_4class"

plotEnsembleConfusionsIndividual(
    soft_equal_4_results,
    "Soft Voting (equal) – ($exp_name)";
    class_labels = class_labels_4,
    save_dir     = soft_equal_4_dir,
    file_prefix  = soft_equal_4_prefix
)

In [ ]:
soft_weighted_4_results = Ensembles.run_voting_ensembles(
    ensemble_specs_4,
    suites_4["Raw Attributes"].y,
    suites_4["Raw Attributes"].y_test;
    voting       = :soft,
    weights      = soft_weights_4,
    title_prefix = "4-class crime"
)

soft_weighted_4_dir    = joinpath("ensembles", exp_name, "soft_weighted_voting")
soft_weighted_4_prefix = "soft_weighted_voting_4class"

plotEnsembleConfusionsIndividual(
    soft_weighted_4_results,
    "Soft Voting (weighted) – ($exp_name)";
    class_labels = class_labels_4,
    save_dir     = soft_weighted_4_dir,
    file_prefix  = soft_weighted_4_prefix
)

In [ ]:
stacking_4_results = Ensembles.run_stacking_ensembles(
    ensemble_specs_4,
    suites_4["Raw Attributes"].y,
    suites_4["Raw Attributes"].y_test;
    base_names   = (:SVM, :DT, :KNN),
    title_prefix = "4-class crime"
)

stacking_4_dir    = joinpath("ensembles", exp_name, "stacking")
stacking_4_prefix = "stacking_4class"

plotEnsembleConfusionsIndividual(
    stacking_4_results,
    "Stacking – ($exp_name)";
    class_labels = class_labels_4,
    save_dir     = stacking_4_dir,
    file_prefix  = stacking_4_prefix
)

In [ ]:
simple_hard_voting_4_results = Ensembles.run_voting_ensembles(
    simple_ensemble_specs,
    suites_4["Raw Attributes"].y,
    suites_4["Raw Attributes"].y_test;
    voting       = :hard,
    title_prefix = "4-class crime (simple)"
)

simple_hard_4_save_dir    = joinpath("ensembles", exp_name, "simple_hard_voting")
simple_hard_4_file_prefix = "simple_hard_voting_4class"

plotEnsembleConfusionsIndividual(
    simple_hard_voting_4_results,
    "Hard Voting (simple) – ($exp_name)";
    class_labels = class_labels_4,
    save_dir     = simple_hard_4_save_dir,
    file_prefix  = simple_hard_4_file_prefix
)

In [ ]:
simple_soft_equal_4_results = Ensembles.run_voting_ensembles(
    simple_ensemble_specs,
    suites_4["Raw Attributes"].y,
    suites_4["Raw Attributes"].y_test;
    voting       = :soft,
    weights      = nothing,
    title_prefix = "4-class crime (simple)"
)

simple_soft_equal_4_dir    = joinpath("ensembles", exp_name, "simple_soft_equal_voting")
simple_soft_equal_4_prefix = "simple_soft_equal_voting_4class"

plotEnsembleConfusionsIndividual(
    simple_soft_equal_4_results,
    "Soft Voting (equal, simple) – ($exp_name)";
    class_labels = class_labels_4,
    save_dir     = simple_soft_equal_4_dir,
    file_prefix  = simple_soft_equal_4_prefix
)

In [ ]:
simple_soft_weighted_4_results = Ensembles.run_voting_ensembles(
    simple_ensemble_specs,
    suites_4["Raw Attributes"].y,
    suites_4["Raw Attributes"].y_test;
    voting       = :soft,
    weights      = soft_weights_4_simple,
    title_prefix = "4-class crime (simple)"
)

simple_soft_weighted_4_dir    = joinpath("ensembles", exp_name, "simple_soft_weighted_voting")
simple_soft_weighted_4_prefix = "simple_soft_weighted_voting_4class"

plotEnsembleConfusionsIndividual(
    simple_soft_weighted_4_results,
    "Soft Voting (weighted, simple) – ($exp_name)";
    class_labels = class_labels_4,
    save_dir     = simple_soft_weighted_4_dir,
    file_prefix  = simple_soft_weighted_4_prefix
)

In [ ]:
simple_stacking_4_results = Ensembles.run_stacking_ensembles(
    simple_ensemble_specs,
    suites_4["Raw Attributes"].y,
    suites_4["Raw Attributes"].y_test;
    base_names   = (:ANN, :SVM, :DT, :KNN),
    title_prefix = "4-class crime (simple)"
)

simple_stacking_4_dir    = joinpath("ensembles", exp_name, "simple_stacking")
simple_stacking_4_prefix = "simple_stacking_4class"

plotEnsembleConfusionsIndividual(
    simple_stacking_4_results,
    "Stacking (simple) – ($exp_name)";
    class_labels = class_labels_4,
    save_dir     = simple_stacking_4_dir,
    file_prefix  = simple_stacking_4_prefix
)

In [ ]:
data_to_plot_ensembles_4 = [
    (
        ds_name,
        [
            ("Hard Voting Ensemble",
                hard_voting_4_results[ds_name][1],
                hard_voting_4_results[ds_name][2]),
            ("Soft Voting Ensemble (Equal)",
                soft_equal_4_results[ds_name][1],
                soft_equal_4_results[ds_name][2]),
            ("Soft Voting Ensemble (Weighted)",
                soft_weighted_4_results[ds_name][1],
                soft_weighted_4_results[ds_name][2]),
            ("Stacking Ensemble",
                stacking_4_results[ds_name][1],
                stacking_4_results[ds_name][2])
        ]
    )
    for ds_name in ordered_datasets
]

data_to_plot_ensembles_4_simple = [
    (
        ds_name,
        [
            ("Hard Voting Ensemble",
                simple_hard_voting_4_results[ds_name][1],
                simple_hard_voting_4_results[ds_name][2]),
            ("Soft Voting Ensemble (Equal)",
                simple_soft_equal_4_results[ds_name][1],
                simple_soft_equal_4_results[ds_name][2]),
            ("Soft Voting Ensemble (Weighted)",
                simple_soft_weighted_4_results[ds_name][1],
                simple_soft_weighted_4_results[ds_name][2]),
            ("Stacking Ensemble",
                simple_stacking_4_results[ds_name][1],
                simple_stacking_4_results[ds_name][2])
        ]
    )
    for ds_name in ordered_datasets
]


plotEnsembleAccuracyBestVsSimple(
    data_to_plot_ensembles_4,
    data_to_plot_ensembles_4_simple;
    folder_name   = "ensembles/4-class/accuracy_best_vs_simple",
    base_filename = "accuracy_4class_best_vs_simple",
    size          = (1200, 600),
    ylims         = (0.5, 0.65) 
)


In [ ]:
class_labels_3 = ["1-low crime \n(0.0, 0.09)", "2- medium crime \n(0.09, 0.25)", "3- high crime \n(0.25, 1.0)"]

exp_name = "3-class"

soft_weights_3_simple = [1.5, 2, 1.3, 1.7]
soft_weights_3 = [2, 1.3, 1.7]


ensemble_specs_3 = create_ensemble_specs(suites_3)



In [ ]:

ensemble_specs_3 = Dict{String, NamedTuple}(k => v for (k,v) in ensemble_specs_3)

hard_voting_3_results = Ensembles.run_voting_ensembles(
    ensemble_specs_3,
    suites_3["Raw Attributes"].y,
    suites_3["Raw Attributes"].y_test;
    voting       = :hard,
    title_prefix = "3-class crime"
)

hard_3_save_dir    = joinpath("ensembles", exp_name, "hard_voting")
hard_3_file_prefix = "hard_voting_3class"

plotEnsembleConfusionsIndividual(
    hard_voting_3_results,
    "Hard Voting – ($exp_name)";
    class_labels = class_labels_3,
    save_dir     = hard_3_save_dir,
    file_prefix  = hard_3_file_prefix
)

In [ ]:
soft_equal_3_results = Ensembles.run_voting_ensembles(
    ensemble_specs_3,
    suites_3["Raw Attributes"].y,
    suites_3["Raw Attributes"].y_test;
    voting       = :soft,
    weights      = nothing,
    title_prefix = "3-class crime"
)

soft_equal_3_dir    = joinpath("ensembles", exp_name, "soft_equal_voting")
soft_equal_3_prefix = "soft_equal_voting_3class"

plotEnsembleConfusionsIndividual(
    soft_equal_3_results,
    "Soft Voting (equal) – ($exp_name)";
    class_labels = class_labels_3,
    save_dir     = soft_equal_3_dir,
    file_prefix  = soft_equal_3_prefix
)

In [ ]:
soft_weighted_3_results = Ensembles.run_voting_ensembles(
    ensemble_specs_3,
    suites_3["Raw Attributes"].y,
    suites_3["Raw Attributes"].y_test;
    voting       = :soft,
    weights      = soft_weights_3,
    title_prefix = "3-class crime"
)

soft_weighted_3_dir    = joinpath("ensembles", exp_name, "soft_weighted_voting")
soft_weighted_3_prefix = "soft_weighted_voting_3class"

plotEnsembleConfusionsIndividual(
    soft_weighted_3_results,
    "Soft Voting (weighted) – ($exp_name)";
    class_labels = class_labels_3,
    save_dir     = soft_weighted_3_dir,
    file_prefix  = soft_weighted_3_prefix
)

In [ ]:
stacking_3_results = Ensembles.run_stacking_ensembles(
    ensemble_specs_3,
    suites_3["Raw Attributes"].y,
    suites_3["Raw Attributes"].y_test;
    base_names   = (:SVM, :DT, :KNN),
    title_prefix = "3-class crime"
)

stacking_3_dir    = joinpath("ensembles", exp_name, "stacking")
stacking_3_prefix = "stacking_3class"

plotEnsembleConfusionsIndividual(
    stacking_3_results,
    "Stacking – ($exp_name)";
    class_labels = class_labels_3,
    save_dir     = stacking_3_dir,
    file_prefix  = stacking_3_prefix
)

In [ ]:
simple_hard_voting_3_results = Ensembles.run_voting_ensembles(
    simple_ensemble_specs,
    suites_3["Raw Attributes"].y,
    suites_3["Raw Attributes"].y_test;
    voting       = :hard,
    title_prefix = "3-class crime (simple)"
)

simple_hard_3_save_dir    = joinpath("ensembles", exp_name, "simple_hard_voting")
simple_hard_3_file_prefix = "simple_hard_voting_3class"

plotEnsembleConfusionsIndividual(
    simple_hard_voting_3_results,
    "Hard Voting (simple) – ($exp_name)";
    class_labels = class_labels_3,
    save_dir     = simple_hard_3_save_dir,
    file_prefix  = simple_hard_3_file_prefix
)

In [ ]:
simple_soft_equal_3_results = Ensembles.run_voting_ensembles(
    simple_ensemble_specs,
    suites_3["Raw Attributes"].y,
    suites_3["Raw Attributes"].y_test;
    voting       = :soft,
    weights      = nothing,
    title_prefix = "3-class crime (simple)"
)

simple_soft_equal_3_dir    = joinpath("ensembles", exp_name, "simple_soft_equal_voting")
simple_soft_equal_3_prefix = "simple_soft_equal_voting_3class"

plotEnsembleConfusionsIndividual(
    simple_soft_equal_3_results,
    "Soft Voting (equal, simple) – ($exp_name)";
    class_labels = class_labels_3,
    save_dir     = simple_soft_equal_3_dir,
    file_prefix  = simple_soft_equal_3_prefix
)

In [ ]:
simple_soft_weighted_3_results = Ensembles.run_voting_ensembles(
    simple_ensemble_specs,
    suites_3["Raw Attributes"].y,
    suites_3["Raw Attributes"].y_test;
    voting       = :soft,
    weights      = soft_weights_3_simple,
    title_prefix = "3-class crime (simple)"
)

simple_soft_weighted_3_dir    = joinpath("ensembles", exp_name, "simple_soft_weighted_voting")
simple_soft_weighted_3_prefix = "simple_soft_weighted_voting_3class"

plotEnsembleConfusionsIndividual(
    simple_soft_weighted_3_results,
    "Soft Voting (weighted, simple) – ($exp_name)";
    class_labels = class_labels_3,
    save_dir     = simple_soft_weighted_3_dir,
    file_prefix  = simple_soft_weighted_3_prefix
)

In [ ]:
simple_stacking_3_results = Ensembles.run_stacking_ensembles(
    simple_ensemble_specs,
    suites_3["Raw Attributes"].y,
    suites_3["Raw Attributes"].y_test;
    base_names   = (:ANN, :SVM, :DT, :KNN),
    title_prefix = "3-class crime (simple)"
)

simple_stacking_3_dir    = joinpath("ensembles", exp_name, "simple_stacking")
simple_stacking_3_prefix = "simple_stacking_3class"

plotEnsembleConfusionsIndividual(
    simple_stacking_3_results,
    "Stacking (simple) – ($exp_name)";
    class_labels = class_labels_3,
    save_dir     = simple_stacking_3_dir,
    file_prefix  = simple_stacking_3_prefix
)

In [ ]:
data_to_plot_ensembles_3 = [
    (
        ds_name,
        [
            ("Hard Voting Ensemble",
                hard_voting_3_results[ds_name][1],
                hard_voting_3_results[ds_name][2]),
            ("Soft Voting Ensemble (Equal)",
                soft_equal_3_results[ds_name][1],
                soft_equal_3_results[ds_name][2]),
            ("Soft Voting Ensemble (Weighted)",
                soft_weighted_3_results[ds_name][1],
                soft_weighted_3_results[ds_name][2]),
            ("Stacking Ensemble",
                stacking_3_results[ds_name][1],
                stacking_3_results[ds_name][2])
        ]
    )
    for ds_name in ordered_datasets
]

data_to_plot_ensembles_3_simple = [
    (
        ds_name,
        [
            ("Hard Voting Ensemble",
                simple_hard_voting_3_results[ds_name][1],
                simple_hard_voting_3_results[ds_name][2]),
            ("Soft Voting Ensemble (Equal)",
                simple_soft_equal_3_results[ds_name][1],
                simple_soft_equal_3_results[ds_name][2]),
            ("Soft Voting Ensemble (Weighted)",
                simple_soft_weighted_3_results[ds_name][1],
                simple_soft_weighted_3_results[ds_name][2]),
            ("Stacking Ensemble",
                simple_stacking_3_results[ds_name][1],
                simple_stacking_3_results[ds_name][2])
        ]
    )
    for ds_name in ordered_datasets
]

plotEnsembleAccuracyBestVsSimple(
    data_to_plot_ensembles_3,
    data_to_plot_ensembles_3_simple;
    folder_name   = "ensembles/3-class/accuracy_best_vs_simple",
    base_filename = "accuracy_3class_best_vs_simple",
    size          = (1200, 600),
    ylims         = (0.6, 0.75) 
)


In [ ]:
# Global Attributes for Experiments
exp_name = "2-class"

class_labels_2 = [
    "1-low crime\n(0.0, 0.15)",
    "2-high crime\n(0.15, 1.0)"
]

soft_weights = [2, 1.3, 1.7]
soft_weights_simple = [1.5, 2, 1.3, 1.7]

ensemble_specs = create_ensemble_specs(suites_2)



In [ ]:
# 1) Run ensembles
ensemble_specs = Dict{String, NamedTuple}(k => v for (k,v) in ensemble_specs)

hard_voting_results = Ensembles.run_voting_ensembles(
    ensemble_specs,
    suites_2["Raw Attributes"].y,
    suites_2["Raw Attributes"].y_test;
    voting       = :hard,
    title_prefix = "2-class crime"
)

# 2) Images directory
save_dir = joinpath("ensembles", exp_name, "hard_voting")
file_prefix = "hard_voting_2class"

# 3) Plots
plotEnsembleConfusionsIndividual(
    hard_voting_results,
    "Hard Voting – ($exp_name)";
    class_labels = class_labels_2,
    save_dir     = save_dir,
    file_prefix  = file_prefix
)


In [ ]:
soft_equal_results = Ensembles.run_voting_ensembles(
    ensemble_specs,
    suites_2["Raw Attributes"].y,
    suites_2["Raw Attributes"].y_test;
    voting       = :soft,           # <- soft
    weights      = nothing,         # <- pesos iguales por defecto
    title_prefix = "2-class crime"
)

soft_equal_dir    = joinpath("ensembles", exp_name, "soft_equal_voting")
soft_equal_prefix = "soft_equal_voting_2class"

plotEnsembleConfusionsIndividual(
    soft_equal_results,
    "Soft Voting (equal) – ($exp_name)";
    class_labels = class_labels_2,
    save_dir     = soft_equal_dir,
    file_prefix  = soft_equal_prefix
)

In [ ]:
soft_weighted_results = Ensembles.run_voting_ensembles(
    ensemble_specs,
    suites_2["Raw Attributes"].y,
    suites_2["Raw Attributes"].y_test;
    voting       = :soft,
    weights      = soft_weights,       
    title_prefix = "2-class crime"
)

soft_weighted_dir    = joinpath("ensembles", exp_name, "soft_weighted_voting")
soft_weighted_prefix = "soft_weighted_voting_2class"

plotEnsembleConfusionsIndividual(
    soft_weighted_results,
    "Soft Voting (weighted) – ($exp_name)";
    class_labels = class_labels_2,
    save_dir     = soft_weighted_dir,
    file_prefix  = soft_weighted_prefix
)

In [ ]:
stacking_results = Ensembles.run_stacking_ensembles(
    ensemble_specs,
    suites_2["Raw Attributes"].y,
    suites_2["Raw Attributes"].y_test;
    base_names   = (:SVM, :DT, :KNN),
    title_prefix = "2-class crime"
)

stacking_dir    = joinpath("ensembles", exp_name, "stacking")
stacking_prefix = "stacking_2class"

plotEnsembleConfusionsIndividual(
    stacking_results,
    "Stacking – ($exp_name)";
    class_labels = class_labels_2,
    save_dir     = stacking_dir,
    file_prefix  = stacking_prefix
)

In [ ]:
simple_hard_voting_results = Ensembles.run_voting_ensembles(
    simple_ensemble_specs,
    suites_2["Raw Attributes"].y,
    suites_2["Raw Attributes"].y_test;
    voting       = :hard,
    title_prefix = "2-class crime (simple)"
)

simple_hard_save_dir    = joinpath("ensembles", exp_name, "simple_hard_voting")
simple_hard_file_prefix = "simple_hard_voting_2class"

plotEnsembleConfusionsIndividual(
    simple_hard_voting_results,
    "Hard Voting (simple) – ($exp_name)";
    class_labels = class_labels_2,
    save_dir     = simple_hard_save_dir,
    file_prefix  = simple_hard_file_prefix
)

In [ ]:
simple_soft_equal_results = Ensembles.run_voting_ensembles(
    simple_ensemble_specs,
    suites_2["Raw Attributes"].y,
    suites_2["Raw Attributes"].y_test;
    voting       = :soft,
    weights      = nothing,                       # igualitario
    title_prefix = "2-class crime (simple)"
)

simple_soft_equal_save_dir    = joinpath("ensembles", exp_name, "simple_soft_equal_voting")
simple_soft_equal_file_prefix = "simple_soft_equal_voting_2class"

plotEnsembleConfusionsIndividual(
    simple_soft_equal_results,
    "Soft Voting (equal, simple) – ($exp_name)";
    class_labels = class_labels_2,
    save_dir     = simple_soft_equal_save_dir,
    file_prefix  = simple_soft_equal_file_prefix
)

In [ ]:
simple_soft_weighted_results = Ensembles.run_voting_ensembles(
    simple_ensemble_specs,
    suites_2["Raw Attributes"].y,
    suites_2["Raw Attributes"].y_test;
    voting       = :soft,
    weights      = soft_weights_simple,
    title_prefix = "2-class crime (simple)"
)

simple_soft_weighted_save_dir    = joinpath("ensembles", exp_name, "simple_soft_weighted_voting")
simple_soft_weighted_file_prefix = "simple_soft_weighted_voting_2class"

plotEnsembleConfusionsIndividual(
    simple_soft_weighted_results,
    "Soft Voting (weighted, simple) – ($exp_name)";
    class_labels = class_labels_2,
    save_dir     = simple_soft_weighted_save_dir,
    file_prefix  = simple_soft_weighted_file_prefix
)

In [ ]:
simple_stacking_results = Ensembles.run_stacking_ensembles(
    simple_ensemble_specs,
    suites_2["Raw Attributes"].y,
    suites_2["Raw Attributes"].y_test;
    base_names   = (:ANN, :SVM, :DT, :KNN),
    title_prefix = "2-class crime"
)

simple_stacking_dir    = joinpath("ensembles", exp_name, "simple_stacking")
simple_stacking_prefix = "simple_stacking_2class"

plotEnsembleConfusionsIndividual(
    simple_stacking_results,
    "Stacking (simple) – ($exp_name)";
    class_labels = class_labels_2,
    save_dir     = simple_stacking_dir,
    file_prefix  = simple_stacking_prefix
)

In [ ]:
# 3) Accuracy comparison: hard + soft + weighted + stacking
data_to_plot_ensembles = [
    (
        ds_name,
        [
            ("Hard Voting Ensemble",
                hard_voting_results[ds_name][1],
                hard_voting_results[ds_name][2]),
            ("Soft Voting Ensemble (Equal)",
                soft_equal_results[ds_name][1],
                soft_equal_results[ds_name][2]),
            ("Soft Voting Ensemble (Weighted)",
                soft_weighted_results[ds_name][1],
                soft_weighted_results[ds_name][2]),
            ("Stacking Ensemble",
                stacking_results[ds_name][1],
                stacking_results[ds_name][2])
        ]
    )
    for ds_name in ordered_datasets
]

data_to_plot_ensembles_simples = [
    (
        ds_name,
        [
            ("Hard Voting Ensemble",
                simple_hard_voting_results[ds_name][1],
                simple_hard_voting_results[ds_name][2]),
            ("Soft Voting Ensemble (Equal)",
                simple_soft_equal_results[ds_name][1],
                simple_soft_equal_results[ds_name][2]),
            ("Soft Voting Ensemble (Weighted)",
                simple_soft_weighted_results[ds_name][1],
                simple_soft_weighted_results[ds_name][2]),
            ("Stacking Ensemble",
                simple_stacking_results[ds_name][1],
                simple_stacking_results[ds_name][2])
        ]
    )
    for ds_name in ordered_datasets
]

plotEnsembleAccuracyBestVsSimple(
    data_to_plot_ensembles,
    data_to_plot_ensembles_simples;
    folder_name   = "ensembles/2-class/accuracy_best_vs_simple",
    base_filename = "accuracy_2class_best_vs_simple",
    size          = (1200, 600),
    ylims         = (0.7, 0.9)   # ajusta a ojo si hace falta
)



In [ ]:
println("\nREGRESSION ENSEMBLES – HARD (EQUAL WEIGHTS)")
println("===========================================")

reg_ensemble_results_hard = Ensembles.run_regression_ensembles(
    reg_ensemble_specs,
    suites["Raw Attributes"].y,
    suites["Raw Attributes"].y_test;
    weights  = nothing
)

reg_ensembles_table_hard = print_reg_ensemble_summary(
    reg_ensemble_results_hard,
    ordered_datasets;
    metric_fn = evaluate_regression,
    title     = "FINAL SUMMARY – Hard Regression Ensemble (Test RMSE)",
    col_label = "Hard Ensemble RMSE"
)


In [ ]:
println("\nREGRESSION ENSEMBLES – EQUAL WEIGHTS")
println("====================================")

reg_ensemble_results_equal = Ensembles.run_regression_ensembles(
    reg_ensemble_specs,
    suites["Raw Attributes"].y,
    suites["Raw Attributes"].y_test;
    weights  = nothing
)

reg_ensembles_table_equal = print_reg_ensemble_summary(
    reg_ensemble_results_equal,
    ordered_datasets;
    metric_fn = evaluate_regression,
    title     = "FINAL SUMMARY – Regression Ensemble (Equal Weights) – Test RMSE",
    col_label = "Equal Weights RMSE"
)

In [ ]:
# ==========================================================
# 2) REGRESSION ENSEMBLE – WEIGHTED (RMSE-based weights)
# ==========================================================
# We use the inverse of the single-model RMSEs as weights,
# so better (lower RMSE) models get higher weight.
# They come from data of cross validation o raw attribute
rmse_ann = 0.1375
rmse_svm = 0.1336
rmse_dt = 0.1349
rmse_knn = 0.1662
base_rmse = [rmse_ann, rmse_svm, rmse_dt, rmse_knn]     
inv_rmse  = 1.0 ./ base_rmse
reg_weights = inv_rmse ./ sum(inv_rmse)

In [ ]:
# ==========================================================
# REGRESSION ENSEMBLES – WEIGHTED AVERAGE
# ==========================================================

println("\nREGRESSION ENSEMBLES – WEIGHTED")
println("================================")

reg_ensemble_results_weighted = Ensembles.run_regression_ensembles(
    reg_ensemble_specs,
    suites["Raw Attributes"].y,
    suites["Raw Attributes"].y_test;
    weights = reg_weights
)

reg_ensembles_table_weighted = print_reg_ensemble_summary(
    reg_ensemble_results_weighted,
    ordered_datasets;
    metric_fn = evaluate_regression,
    title     = "FINAL SUMMARY – Regression Ensemble (Weighted) – Test RMSE",
    col_label = "Weighted RMSE"
)


In [ ]:
reg_stacking_results = Ensembles.run_regression_stacking_ensembles(
    reg_ensemble_specs,         
    suites["Raw Attributes"].y,
    suites["Raw Attributes"].y_test;
    base_names     = (:ANN, :SVM, :DT, :KNN),
    meta_regressor = NeuralNetworkRegressor(
        builder=MLJFlux.MLP((8,8), Flux.relu), 
        epochs=8, 
        batch_size=32,
        optimiser=Optimisers.Adam(0.01) 
    ),
    nfolds         = 5,
    rng_seed       = 123,
    title_prefix   = "Stacking Regression (full)"
)

reg_ensembles_table_stack = print_reg_ensemble_summary(
    reg_stacking_results,
    ordered_datasets;
    metric_fn = evaluate_regression,
    title     = "FINAL SUMMARY – Regression Stacking Ensemble (Full Models, Test RMSE)",
    col_label = "Stacking RMSE (full)"
)

In [ ]:
println("\nSIMPLE REGRESSION ENSEMBLES – HARD (EQUAL WEIGHTS)")
println("====================================================")

simple_reg_ensemble_results_hard = Ensembles.run_regression_ensembles(
    simple_reg_ensemble_specs,
    suites["Raw Attributes"].y,
    suites["Raw Attributes"].y_test;
    weights = nothing
)

simple_reg_ensembles_table_hard = print_reg_ensemble_summary(
    simple_reg_ensemble_results_hard,
    ordered_datasets;
    metric_fn = evaluate_regression,
    title     = "FINAL SUMMARY – Simple Hard Regression Ensemble (Test RMSE)",
    col_label = "Simple Hard Ensemble RMSE"
)

In [ ]:
println("\nSIMPLE REGRESSION ENSEMBLES – EQUAL WEIGHTS")
println("===========================================")

simple_reg_ensemble_results_equal = Ensembles.run_regression_ensembles(
    simple_reg_ensemble_specs,
    suites["Raw Attributes"].y,
    suites["Raw Attributes"].y_test;
    weights = nothing
)

simple_reg_ensembles_table_equal = print_reg_ensemble_summary(
    simple_reg_ensemble_results_equal,
    ordered_datasets;
    metric_fn = evaluate_regression,
    title     = "FINAL SUMMARY – Simple Regression Ensemble (Equal Weights) – Test RMSE",
    col_label = "Simple Equal Weights RMSE"
)

In [ ]:
simple_reg_ensemble_results_weighted = Ensembles.run_regression_ensembles(
    simple_reg_ensemble_specs,
    suites["Raw Attributes"].y,
    suites["Raw Attributes"].y_test;
    weights = reg_weights        
)

simple_reg_ensembles_table_weighted = print_reg_ensemble_summary(
    simple_reg_ensemble_results_weighted,
    ordered_datasets;
    metric_fn = evaluate_regression,
    title     = "FINAL SUMMARY – Simple Regression Ensemble (Weighted) – Test RMSE",
    col_label = "Simple Weighted RMSE"
)

In [ ]:
println("\nSIMPLE REGRESSION ENSEMBLES – STACKING")
println("======================================")

simple_reg_stacking_results = Ensembles.run_regression_stacking_ensembles(
    simple_reg_ensemble_specs,
    suites["Raw Attributes"].y,
    suites["Raw Attributes"].y_test;
    base_names     = (:ANN, :SVM, :DT, :KNN),
    meta_regressor = NeuralNetworkRegressor(),
    nfolds       = 5,
    rng_seed     = 123,
    title_prefix = "Stacking Regression (simple)"
)

simple_reg_ensembles_table_stack = print_reg_ensemble_summary(
    simple_reg_stacking_results,
    ordered_datasets;
    metric_fn = evaluate_regression,
    title     = "FINAL SUMMARY – Simple Regression Stacking Ensemble (Test RMSE)",
    col_label = "Simple Stacking RMSE"
)

In [ ]:
summary_hard     = Ensembles.summarize_regression_ensembles(reg_ensemble_results_hard,     ordered_datasets; metric_fn=evaluate_regression)
summary_equal    = Ensembles.summarize_regression_ensembles(reg_ensemble_results_equal,    ordered_datasets; metric_fn=evaluate_regression)
summary_weighted = Ensembles.summarize_regression_ensembles(reg_ensemble_results_weighted, ordered_datasets; metric_fn=evaluate_regression)
summary_stacking = Ensembles.summarize_regression_ensembles(reg_stacking_results,          ordered_datasets; metric_fn=evaluate_regression)

simple_summary_hard = Ensembles.summarize_regression_ensembles(simple_reg_ensemble_results_hard,            ordered_datasets; metric_fn = evaluate_regression)
simple_summary_equal = Ensembles.summarize_regression_ensembles(simple_reg_ensemble_results_equal,          ordered_datasets; metric_fn = evaluate_regression)
simple_summary_weighted = Ensembles.summarize_regression_ensembles(simple_reg_ensemble_results_weighted,    ordered_datasets; metric_fn = evaluate_regression)
simple_summary_stacking = Ensembles.summarize_regression_ensembles(simple_reg_stacking_results,             ordered_datasets; metric_fn = evaluate_regression)


plot_regression_ensemble_best_vs_simple(
    summary_hard,
    summary_equal,
    summary_weighted,
    summary_stacking,
    simple_summary_hard,
    simple_summary_equal,
    simple_summary_weighted,
    simple_summary_stacking,
    ordered_datasets;
    ensemble_labels = ["Hard Voting", "Equal Weights", "Weighted", "Stacking"],
    folder_name     = "ensembles/regression/",
    base_filename   = "regression_best_vs_simple",
    size            = (1200, 600),
    ylims           = (0, 0.5)   # o por ejemplo (0.4, 0.9)
)

